## Receive the parameter

In [0]:
from pyspark.sql.functions import *
from datetime import date

dbutils.widgets.text("selected_date", "")

selected_date = dbutils.widgets.get("selected_date")

if not selected_date:
    raise ValueError("Missing selected_date parameter")

selected_date = date.fromisoformat(selected_date)

## Extract the transactions

In [0]:
source_table = (
    "retail_project.external_app_schema.future_transactions"
)

df_daily = (
    spark.table(source_table)
    .filter(col("transaction_date") == lit(selected_date))
    .withColumnRenamed("customerID", "Customer ID")
    .select(
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    )
)

row_count = df_daily.count()

if row_count == 0:
    raise RuntimeError(f"No transactions for {selected_date}")

print(f"Publishing {row_count:,} rows for {selected_date}")

## Write the csv to a temporary location

In [0]:
temp_directory = (
    "/Volumes/retail_project/external_app_schema/"
    f"external_app_volume/temp_publish/{selected_date}/"
)

(
    df_daily
    .coalesce(1)
    .write
    .format("csv")
    .option("header", "true")
    .mode("overwrite")
    .save(temp_directory)
)

## Publish it

In [0]:
from pathlib import Path

landing_path = (
    "/Volumes/retail_project/landing/raw_files/"
)

filename = f"transactions_{selected_date}.csv"
destination = landing_path + filename

part_files = [
    item.path
    for item in dbutils.fs.ls(temp_directory)
    if item.name.startswith("part-")
    and item.name.endswith(".csv")
]

if len(part_files) != 1:
    raise RuntimeError(
        f"Expected one part file, found {len(part_files)}"
    )

if Path(destination).exists():
    # A previous attempt may have published the file
    # before the control table was updated.
    print(f"Destination already exists: {destination}")
else:
    if not dbutils.fs.mv(part_files[0], destination):
        raise RuntimeError("Publication failed")

if not Path(destination).exists():
    raise RuntimeError("Published file not found")

dbutils.fs.rm(temp_directory, recurse=True)

print(f"Published: {destination}")

## Pass publication metadata onward

In [0]:
dbutils.jobs.taskValues.set(
    key="selected_date",
    value=selected_date.isoformat()
)

dbutils.jobs.taskValues.set(
    key="file_path",
    value=destination
)

dbutils.jobs.taskValues.set(
    key="row_count",
    value=row_count
)